In [1]:
import numpy as np
from collections import Counter
import random

# ------------------------------
# 1. Decision Tree Node
# ------------------------------
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature        # Feature index to split on
        self.threshold = threshold    # Threshold value for split
        self.left = left              # Left child
        self.right = right            # Right child
        self.value = value            # Final class if it's a leaf node

    def is_leaf_node(self):
        return self.value is not None


# ------------------------------
# 2. Decision Tree Classifier
# ------------------------------
class DecisionTree:
    def __init__(self, max_depth=10, min_samples_split=2, n_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features
        self.root = None

    def fit(self, X, y):
        self.n_features = X.shape[1] if self.n_features is None else min(self.n_features, X.shape[1])
        self.root = self._grow_tree(X, y)

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        feat_idxs = np.random.choice(n_features, self.n_features, replace=False)
        best_feat, best_thresh = self._best_split(X, y, feat_idxs)

        left_idxs = X[:, best_feat] < best_thresh
        right_idxs = ~left_idxs

        left = self._grow_tree(X[left_idxs], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs], y[right_idxs], depth + 1)
        return Node(best_feat, best_thresh, left, right)

    def _best_split(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_thresh = None, None

        for feat_idx in feat_idxs:
            thresholds = np.unique(X[:, feat_idx])
            for thresh in thresholds:
                gain = self._gini_gain(y, X[:, feat_idx], thresh)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = thresh

        return split_idx, split_thresh

    def _gini_gain(self, y, feature_column, threshold):
        left_idxs = feature_column < threshold
        right_idxs = ~left_idxs

        if len(y[left_idxs]) == 0 or len(y[right_idxs]) == 0:
            return 0

        def gini(labels):
            counts = np.bincount(labels)
            probs = counts / len(labels)
            return 1 - np.sum(probs ** 2)

        parent_gini = gini(y)
        left_gini = gini(y[left_idxs])
        right_gini = gini(y[right_idxs])
        n = len(y)

        weighted_gini = (len(y[left_idxs]) / n) * left_gini + (len(y[right_idxs]) / n) * right_gini
        return parent_gini - weighted_gini

    def _most_common_label(self, y):
        return Counter(y).most_common(1)[0][0]

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value
        if x[node.feature] < node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)


# ------------------------------
# 3. Random Forest Classifier
# ------------------------------
class RandomForest:
    def __init__(self, n_trees=10, max_depth=10, min_samples_split=2, n_features=None):
        self.n_trees = n_trees
        self.trees = []
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features

    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_trees):
            idxs = np.random.choice(len(X), len(X), replace=True)
            X_sample, y_sample = X[idxs], y[idxs]
            tree = DecisionTree(max_depth=self.max_depth,
                                min_samples_split=self.min_samples_split,
                                n_features=self.n_features)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        return np.array([self._most_common_label(tree_preds[:, i]) for i in range(X.shape[0])])

    def _most_common_label(self, labels):
        return Counter(labels).most_common(1)[0][0]


In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load dataset
data = load_iris()
X, y = data.data, data.target

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Train Random Forest
clf = RandomForest(n_trees=10, max_depth=10, n_features=2)
clf.fit(X_train, y_train)

# Predict and evaluate
predictions = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, predictions))


Accuracy: 0.9
